In [1]:
import os
import json
from tqdm import tqdm

def read_json_file(path: str) -> dict | list:
    with open(path, 'r') as f:
        return json.load(f)

def write_json_file(data: dict | list, path: str) -> None:
    with open(path, 'w') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

In [2]:
# # Install transformers if needed and load Qwen tokenizer
# from transformers import AutoTokenizer
# from copy import deepcopy

# # Load Qwen tokenizer
# tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B", trust_remote_code=True)

## 1. QueryRefiner

In [3]:
qr_raw_data = []
for file in tqdm(os.listdir("./data/raw_data/cluster_samples_QR")):
    if file.endswith(".json"):
        tmp_data = read_json_file(os.path.join("./data/raw_data/cluster_samples_QR", file))
        for d in tmp_data:
            if "input_data" in d and "output_data" in d:
                d["cluster"] = int(file.split(".")[0].replace("cluster_", ""))
                d["input_data"] = json.loads(d["input_data"])
                d["output_data"] = json.loads(d["output_data"])
                qr_raw_data.append(d)

100%|██████████| 20/20 [00:00<00:00, 52.46it/s]


In [6]:
qr_stats = {}
for d in tqdm(qr_raw_data):
    qr_stats[d["cluster"]] = qr_stats.get(d["cluster"], 0) + 1

qr_stats = dict(sorted(qr_stats.items(), key=lambda item: item[0], reverse=False))
qr_stats

100%|██████████| 7836/7836 [00:00<00:00, 866140.47it/s]


{0: 300,
 1: 800,
 2: 550,
 3: 100,
 4: 50,
 5: 200,
 6: 500,
 7: 300,
 8: 754,
 9: 200,
 10: 700,
 11: 634,
 12: 600,
 13: 100,
 14: 300,
 15: 600,
 16: 20,
 17: 886,
 19: 242}

## 2. RelevanctDetection

## 3. General Chain

In [12]:
general_raw_data = []
for file in tqdm(os.listdir("./data/raw_data/cluster_samples_Gen")):
    if file.endswith(".json"):
        tmp_data = read_json_file(os.path.join("./data/raw_data/cluster_samples_Gen", file))
        for d in tmp_data:
            if ("input_data" in d) and ("output_data" in d) and (d["chain_name"] == "general_query_generation_chain"):
                d["cluster"] = int(file.split(".")[0].replace("cluster_", ""))
                d["input_data"] = json.loads(d["input_data"])
                d["output_data"] = json.loads(d["output_data"])
                general_raw_data.append(d)

100%|██████████| 20/20 [00:00<00:00, 21.75it/s]


In [15]:
general_stats = {}
for d in tqdm(general_raw_data):
    general_stats[d["cluster"]] = general_stats.get(d["cluster"], 0) + 1

general_stats = dict(sorted(general_stats.items(), key=lambda item: item[0], reverse=False))

100%|██████████| 113/113 [00:00<00:00, 341467.11it/s]


In [16]:
general_stats

{1: 3, 2: 13, 3: 1, 4: 6, 7: 1, 8: 13, 12: 1, 13: 71, 15: 4}

## 4. Unrelated Chain

## 5. Related - QueryTag

In [7]:
qt_raw_data = []
for file in tqdm(os.listdir("./data/raw_data/cluster_samples_QT")):
    if file.endswith(".json"):
        tmp_data = read_json_file(os.path.join("./data/raw_data/cluster_samples_QT", file))
        for d in tmp_data:
            if "input_data" in d and "output_data" in d:
                d["cluster"] = int(file.split(".")[0].replace("cluster_", ""))
                d["input_data"] = json.loads(d["input_data"])
                d["output_data"] = json.loads(d["output_data"])
                qt_raw_data.append(d)

100%|██████████| 20/20 [00:00<00:00, 91.90it/s]


In [8]:
qt_cluster_stats = {}
qt_tag_stats = {}
qt_mixed_stats = {}
for d in tqdm(qt_raw_data):
    qt_cluster_stats[d["cluster"]] = qt_cluster_stats.get(d["cluster"], 0) + 1
    qt_tag_stats[d["output_data"]["tag"]] = qt_tag_stats.get(d["output_data"]["tag"], 0) + 1
    
    if d["cluster"] not in qt_mixed_stats:
        qt_mixed_stats[d["cluster"]] = {}
    qt_mixed_stats[d["cluster"]][d["output_data"]["tag"]] = qt_mixed_stats[d["cluster"]].get(d["output_data"]["tag"], 0) + 1

qt_cluster_stats = dict(sorted(qt_cluster_stats.items(), key=lambda item: item[0], reverse=False))
qt_tag_stats = dict(sorted(qt_tag_stats.items(), key=lambda item: item[0], reverse=False))
qt_mixed_stats = dict(sorted(qt_mixed_stats.items(), key=lambda item: item[0], reverse=False))

100%|██████████| 5653/5653 [00:00<00:00, 899108.89it/s]


In [9]:
qt_cluster_stats

{0: 209,
 1: 614,
 2: 494,
 3: 90,
 4: 44,
 5: 131,
 6: 305,
 7: 213,
 8: 621,
 9: 147,
 10: 557,
 11: 279,
 12: 368,
 13: 83,
 14: 170,
 15: 538,
 16: 15,
 17: 659,
 19: 116}

In [10]:
qt_tag_stats

{'connect_operator': 1556, 'explain_more': 546, 'normal': 3551}

In [11]:
qt_mixed_stats

{0: {'connect_operator': 33, 'normal': 157, 'explain_more': 19},
 1: {'connect_operator': 110, 'normal': 454, 'explain_more': 50},
 2: {'normal': 335, 'connect_operator': 80, 'explain_more': 79},
 3: {'normal': 27, 'connect_operator': 54, 'explain_more': 9},
 4: {'normal': 33, 'connect_operator': 8, 'explain_more': 3},
 5: {'normal': 84, 'explain_more': 22, 'connect_operator': 25},
 6: {'normal': 241, 'connect_operator': 39, 'explain_more': 25},
 7: {'normal': 165, 'explain_more': 30, 'connect_operator': 18},
 8: {'normal': 333, 'connect_operator': 247, 'explain_more': 41},
 9: {'normal': 102, 'connect_operator': 29, 'explain_more': 16},
 10: {'normal': 292, 'explain_more': 60, 'connect_operator': 205},
 11: {'normal': 151, 'connect_operator': 117, 'explain_more': 11},
 12: {'connect_operator': 147, 'normal': 187, 'explain_more': 34},
 13: {'normal': 76, 'connect_operator': 7},
 14: {'normal': 121, 'connect_operator': 37, 'explain_more': 12},
 15: {'explain_more': 69, 'normal': 306, 'c

## 4. Related - Generation

In [28]:
# Prepare Generation Data
generation_raw_data = []
for file in tqdm(os.listdir("./data/raw_data/cluster_samples_Gen")):
    if file.endswith(".json"):
        tmp_data = read_json_file(os.path.join("./data/raw_data/cluster_samples_Gen", file))
        for d in tmp_data:
            if ("input_data" in d) and ("output_data" in d) and (d["chain_name"] != "general_query_generation_chain"):
                d["chain_name"] = d["chain_name"].replace("_query_generation_chain", "")
                d["cluster"] = int(file.split(".")[0].replace("cluster_", ""))
                d["input_data"] = json.loads(d["input_data"])
                d["output_data"] = json.loads(d["output_data"])
                generation_raw_data.append(d)


100%|██████████| 20/20 [00:01<00:00, 17.59it/s]


In [29]:
gen_cluster_stats = {}
gen_chain_stats = {}
gen_mixed_stats = {}
for d in tqdm(generation_raw_data):
    
    cluster = d["cluster"]
    chain = d["chain_name"]
    
    gen_cluster_stats[cluster] = gen_cluster_stats.get(cluster, 0) + 1
    gen_chain_stats[chain] = gen_chain_stats.get(chain, 0) + 1
    
    if cluster not in gen_mixed_stats:
        gen_mixed_stats[cluster] = {}
    
    gen_mixed_stats[cluster][chain] = gen_mixed_stats[cluster].get(chain, 0) + 1

gen_cluster_stats = dict(sorted(gen_cluster_stats.items(), key=lambda item: item[0], reverse=False))
gen_chain_stats = dict(sorted(gen_chain_stats.items(), key=lambda item: item[0], reverse=False))
gen_mixed_stats = dict(sorted(gen_mixed_stats.items(), key=lambda item: item[0], reverse=False))

100%|██████████| 6088/6088 [00:00<00:00, 1209268.93it/s]


In [30]:
gen_cluster_stats

{0: 266,
 1: 681,
 2: 423,
 3: 45,
 4: 27,
 5: 175,
 6: 460,
 7: 281,
 8: 480,
 9: 170,
 10: 494,
 11: 517,
 12: 452,
 13: 20,
 14: 263,
 15: 425,
 16: 9,
 17: 693,
 19: 207}

In [31]:
gen_chain_stats

{'more_explain': 1,
 'more_explain_manual_docs': 537,
 'related': 8,
 'related_manual_docs': 5542}

In [32]:
gen_mixed_stats

{0: {'related_manual_docs': 247, 'more_explain_manual_docs': 19},
 1: {'related_manual_docs': 632, 'more_explain_manual_docs': 48, 'related': 1},
 2: {'related_manual_docs': 345, 'more_explain_manual_docs': 78},
 3: {'related_manual_docs': 35, 'more_explain_manual_docs': 9, 'related': 1},
 4: {'related_manual_docs': 24, 'more_explain_manual_docs': 3},
 5: {'related_manual_docs': 153, 'more_explain_manual_docs': 22},
 6: {'related_manual_docs': 433, 'more_explain_manual_docs': 25, 'related': 2},
 7: {'related_manual_docs': 251, 'more_explain_manual_docs': 30},
 8: {'related_manual_docs': 440, 'more_explain_manual_docs': 39, 'related': 1},
 9: {'related_manual_docs': 154, 'more_explain_manual_docs': 16},
 10: {'related_manual_docs': 435, 'more_explain_manual_docs': 59},
 11: {'related_manual_docs': 506, 'more_explain_manual_docs': 11},
 12: {'related_manual_docs': 418, 'more_explain_manual_docs': 34},
 13: {'related_manual_docs': 20},
 14: {'related_manual_docs': 251, 'more_explain_manua